In [7]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.live_pipeline.nba.min_pipeline import min_pipeline
from src.live_pipeline.nba.apm_pipeline import *
from src.live_pipeline.nba.ppm_pipeline import *
from src.live_pipeline.nba.rpm_pipeline import *
# from src.utils.helpers import *
# from src.utils.dataScraper import *
# from live import *

# warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)


In [2]:
from src.pipeline.fetch import GameLogs

logs = GameLogs("2026", season_type="Regular Season", league="wnba").fetch(
    start_position_delay=2.5,
    start_position_workers=5,
)
logs


Fetching WNBA player_base, player_adv, team_base, team_adv, start_positions for 2026 Regular Season...
✓ team_base → wnba_team_base — 348 rows
    … 348/348 rows
  ✓ raw.wnba_team_base — 348 rows upserted (postgres)
✓ team_adv → wnba_team_adv — 348 rows
    … 348/348 rows
  ✓ raw.wnba_team_adv — 348 rows upserted (postgres)
✓ player_adv → wnba_player_adv — 3,456 rows
    … 3,456/3,456 rows
  ✓ raw.wnba_player_adv — 3,456 rows upserted (postgres)
✓ player_base → wnba_player_base — 3,456 rows
    … 3,456/3,456 rows
  ✓ raw.wnba_player_base — 3,456 rows upserted (postgres)
✓ Checkpoint loaded — 2037/174 games done, 4 remaining

── Batch 1/1 (4 games) ──
  ✓ Batch 1 done — checkpoint saved (46,608 rows total)
✓ start_positions — 4,190 rows (174 games)
    … 4,190/4,190 rows
  ✓ raw.wnba_start_positions — 4,190 rows upserted (postgres)


In [3]:
from src.pipeline.clean import build_silver

silver = build_silver(
    "2026",
    "Regular Season",
    league="wnba",
    raw_frames=logs.data,   # reuse what you just fetched
    db_upsert=True,         # write to silver.wnba_player_gamelogs
)
silver.head()

── Silver (WNBA): 2026 Regular Season ──
  merged — (3456, 184)
  pos tracking fallback: filled 348 / 3,456 missing with PG/SG/SF/PF/C
    … 3,456/3,456 rows
  ✓ silver.wnba_player_gamelogs — 3,456 rows upserted (postgres)
✓ Silver frame — 3,456 rows, 185 columns


,season_year,player_id,player_name,nickname,team_id,team_abbreviation,team_name,game_id,game_date,matchup,wl,min,fgm,fga,fg_pct,fg3m,fg3a,fg3_pct,ftm,fta,ft_pct,oreb,dreb,reb,ast,tov,stl,blk,blka,pf,pfd,pts,plus_minus,nba_fantasy_pts,dd2,td3,wnba_fantasy_pts,available_flag,min_sec,team_count,e_off_rating,off_rating,sp_work_off_rating,e_def_rating,def_rating,sp_work_def_rating,e_net_rating,net_rating,sp_work_net_rating,ast_pct,ast_to,ast_ratio,oreb_pct,dreb_pct,reb_pct,tm_tov_pct,e_tov_pct,efg_pct,ts_pct,usg_pct,e_usg_pct,e_pace,pace,pace_per40,sp_work_pace,pie,poss,fgm_pg,fga_pg,team_city,team_tricode,team_slug,first_name,family_name,name_i,player_slug,start_position,comment,jersey_num,minutes,spd,dist,orbc,drbc,rbc,tchs,sast,ftast,pass,assists,cfgm,cfga,cfg_pct,ufgm,ufga,ufg_pct,field_goal_percentage,dfgm,dfga,dfg_pct,team_fgm,team_fga,team_fg_pct,team_fg3m,team_fg3a,team_fg3_pct,team_ftm,team_fta,team_ft_pct,team_oreb,team_dreb,team_reb,team_ast,team_tov,team_stl,team_blk,team_blka,team_pf,team_pfd,team_pts,team_plus_minus,team_e_off_rating,team_off_rating,team_e_def_rating,team_def_rating,team_e_net_rating,team_net_rating,team_ast_pct,team_ast_to,team_ast_ratio,team_oreb_pct,team_dreb_pct,team_reb_pct,team_tm_tov_pct,team_efg_pct,team_ts_pct,team_e_pace,team_pace,team_pace_per40,team_poss,team_pie,opp_team_id,opp_fgm,opp_fga,opp_fg_pct,opp_fg3m,opp_fg3a,opp_fg3_pct,opp_ftm,opp_fta,opp_ft_pct,opp_oreb,opp_dreb,opp_reb,opp_ast,opp_tov,opp_stl,opp_blk,opp_blka,opp_pf,opp_pfd,opp_pts,opp_plus_minus,opp_e_off_rating,opp_off_rating,opp_e_def_rating,opp_def_rating,opp_e_net_rating,opp_net_rating,opp_ast_pct,opp_ast_to,opp_ast_ratio,opp_oreb_pct,opp_dreb_pct,opp_reb_pct,opp_tm_tov_pct,opp_efg_pct,opp_ts_pct,opp_e_pace,opp_pace,opp_pace_per40,opp_poss,opp_pie,season_type,pos
0,2026,1629477,Sabrina Ionescu,Sabrina,1611661313,NYL,New York Liberty,1022600171,2026-07-12T00:00:00,NYL @ TOR,L,33.116667,9,18,0.500,3,7,0.429,7,8,0.875,1,3,4,8,1,3,0,2,3,7,28,-3,52.8,0,0,49.0,1,33:07,1,96.1,104.2,104.2,103.8,106.9,106.9,-7.8,-2.7,-2.7,0.421,8.00,26.7,0.026,0.111,0.061,3.3,3.3,0.583,0.651,0.253,0.259,109.58,103.63,86.36,103.63,0.265,71,9.0,18.0,New York,NYL,liberty,Sabrina,Ionescu,S. Ionescu,sabrina-ionescu,G,,,33:07,0.0,0.0,0,0,0,0,0,0,0,8,0,0,0.0,0,0,0.0,0.5,0,0,0.0,33,76,0.434,5,26,0.192,20,22,0.909,10,23,33,16,18.0,11,4,3,19,28,91,-2.0,97.1,102.2,101.2,104.5,-4.0,-2.2,0.485,0.89,13.2,0.386,0.750,0.539,0.202,0.467,0.531,111.4,106.8,89.0,89,0.489,1611661332,34,66,0.515,9,24,0.375,16,18,0.889,5,25,30,22,23.0,10,3,4,28,19,93,2.0,101.2,104.5,97.1,102.2,4.0,2.2,0.647,0.96,18.6,0.250,0.614,0.461,0.258,0.583,0.629,111.4,106.8,89.0,89,0.511,Regular Season,None
2,2026,1629567,Natisha Hiedeman,Natisha,1611661328,SEA,Seattle Storm,1022600172,2026-07-12T00:00:00,SEA @ WAS,L,32.895000,14,24,0.583,2,5,0.400,1,1,1.000,0,3,3,3,4,3,0,0,1,1,31,3,44.1,0,0,45.0,1,32:54,1,108.9,109.0,109.0,98.4,104.5,104.5,10.5,4.5,4.5,0.214,0.75,9.7,0.000,0.079,0.043,12.9,12.7,0.625,0.634,0.337,0.347,100.83,97.77,81.47,97.77,0.215,67,14.0,24.0,Seattle,SEA,storm,Natisha,Hiedeman,N. Hiedeman,natisha-hiedeman,G,,,32:54,0.0,0.0,0,0,0,0,0,0,0,3,0,0,0.0,0,0,0.0,0.583,0,0,0.0,30,70,0.429,5,19,0.263,14,18,0.778,16,27,43,12,20.0,12,4,3,25,17,79,-5.0,96.4,96.3,95.3,102.4,1.1,-6.1,0.400,0.60,10.7,0.429,0.660,0.551,0.244,0.464,0.507,102.0,98.4,82.0,82,0.432,1611661322,31,72,0.431,5,19,0.263,17,23,0.739,9,23,32,23,15.0,10,3,4,17,25,84,5.0,95.3,102.4,96.4,96.3,-1.1,6.1,0.742,1.53,19.0,0.340,0.571,0.449,0.183,0.465,0.511,102.0,98.4,82.0,82,0.568,Regular Season,None
5,2026,1641648,Aliyah Boston,Aliyah,1611661325,IND,Indiana Fever,1022600174,2026-07-12T00:00:00,IND @ LVA,W,29.081667,9,14,0.643,1,3,0.333,0,0,0.000,1,10,11,4,1,3,1,1,2,4,19,16,49.2,1,0,43.0,1,29:05,1,121.0,124.1,124.1,101.3,98.2,98.2,19.7,25.9,25.9,0.211,4.00,21.1,0.045,0.345,0.216,5.3,5.3,0.679,0.679,0.238,0.240,94.74,94.91,79.09,94.91,0.266,58,9.0,14.0,Indiana,IND,fever,Aliyah,Boston,A. Boston,aliyah-boston,C,,,29:05,0.0,0.0,0,

### Get updated lineups

In [40]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
No data available. Run getDict() first.
{}

Out Players:
No data available. Run getDict() first.
{}
No data available. Run getDict() first.
No data available. Run getDict() first.


### Dataset

In [8]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
p25 = pd.read_csv('data/raw/playoff_stats/P25.csv').sort_values(by='GAME_DATE')
s25 = pd.concat([s25, p25])


s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
p26 = pd.read_csv('data/raw/playoff_stats/P26.csv').sort_values(by='GAME_DATE')
s26 = pd.concat([s26, p26])

base_df = pd.concat([s25, s26])
print(f"Data contains {base_df.shape[0]} rows and {base_df.shape[1]} columns")
print(base_df.columns[:50])
base_df.head()

/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_33305/1517111039.py:6: DtypeWarning: Columns (186) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')


Data contains 82501 rows and 191 columns
Index(['SEASON_YEAR', 'PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID',
       'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP',
       'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM',
       'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK',
       'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2',
       'TD3', 'WNBA_FANTASY_PTS', 'AVAILABLE_FLAG', 'MIN_SEC', 'TEAM_COUNT',
       'E_OFF_RATING', 'OFF_RATING', 'sp_work_OFF_RATING', 'E_DEF_RATING',
       'DEF_RATING', 'sp_work_DEF_RATING', 'E_NET_RATING', 'NET_RATING',
       'sp_work_NET_RATING', 'AST_PCT'],
      dtype='object')


,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,name,POS,AGE,IS_PLAYOFF,STARTING,PTS_PER_MIN,AST_PER_MIN,REB_PER_MIN,IS_HOME,POSITION_ENCODED,TEAM_SPREAD,GAME_TOTAL,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,TEAM_SPREAD_ODDS,GAME_TOTAL_ODDS
26305,2024-25,1629674,Neemias Queta,Neemias,1610612738,BOS,Boston Celtics,22400061,2024-10-22,BOS vs. NYK,W,4.333333,0,0,0.000,0,0,0.00,0,0,0.0,0,0,0,0,0,0,0,0,1,0,0,0,0.0,0,0,0.0,1,4:20,1,22.2,25.0,25.0,28.6,25.0,25.0,-6.3,0.0,0.0,0.000,0.0,0.0,0.000,0.000,0.000,0.0,0.0,0.000,0.000,0.000,0.000,88.62,88.62,73.85,88.62,0.167,8,0.0,0.0,NaN,3.51,0.28,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.000,0.0,0.0,0.000,0.0,1.0,0.00,48,95,0.505,29,61,0.475,7,8,0.875,11,29,40,33,4.0,6,3,3,15,12,132,23.0,144.2,145.1,118.4,119.8,25.8,25.3,0.688,8.25,23.9,0.298,0.816,0.529,0.044,0.658,0.670,91.8,91.0,75.83,91,0.581,1610612752,NYK,New York Knicks,43,78,0.551,11,30,0.367,12,16,0.750,5,29,34,20,12.0,2,3,3,12,15,109,-23.0,118.4,119.8,144.2,145.1,-25.8,-25.3,0.465,1.67,16.9,0.184,0.702,0.471,0.132,0.622,0.641,91.8,91.0,75.83,91,0.419,Neemias Queta,C,26.0,0,0,0.000000,0.000000,0.000000,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26281,2024-25,1626166,Cameron Payne,Cameron,1610612752,NYK,New York Knicks,22400061,2024-10-22,NYK @ BOS,L,20.716667,5,10,0.500,1,4,0.25,0,0,0.0,1,2,3,4,1,1,0,1,3,0,11,0,22.6,0,0,21.0,1,20:43,1,123.8,124.4,124.4,131.2,130.8,130.8,-7.4,-6.4,-6.4,0.267,4.0,26.7,0.053,0.091,0.073,6.7,6.7,0.550,0.550,0.250,0.255,92.77,92.68,77.23,92.68,0.089,41,5.0,10.0,NaN,4.48,1.65,1.0,3.0,4.0,52.0,0.0,0.0,40.0,3.0,4.0,0.750,2.0,6.0,0.333,2.0,2.0,1.00,43,78,0.551,11,30,0.367,12,16,0.750,5,29,34,20,12.0,2,3,3,12,15,109,-23.0,118.4,119.8,144.2,145.1,-25.8,-25.3,0.465,1.67,16.9,0.184,0.702,0.471,0.132,0.622,0.641,91.8,91.0,75.83,91,0.419,1610612738,BOS,Boston Celtics,48,95,0.505,29,61,0.475,7,8,0.875,11,29,40,33,4.0,6,3,3,15,12,132,23.0,144.2,145.1,118.4,119.8,25.8,25.3,0.688,8.25,23.9,0.298,0.816,0.529,0.044,0.658,0.670,91.8,91.0,75.83,91,0.581,Cameron Payne,PG,31.0,0,0,0.530973,0.193081,0.144811,0,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26280,2024-25,1629638,Nickeil Alexander-Walker,Nickeil,1610612750,MIN,Minnesota Timberwolves,22400062,2024-10-22,MIN @ LAL,L,28.366667,5,7,0.714,3,4,0.75,1,2,0.5,2,3,5,0,2,0,0,0,0,1,14,13,18.0,0,0,22.0,1,28:22,1,130.8,135.2,135.2,105.2,107.1,107.1,25.6,28.0,28.0,0.000,0.0

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_odds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_odds = pd.json_normalize(data)

print("Loaded:", file.name)
team_odds.head()

Loaded: NBA_20260519_103917.json


,home_team,away_team,commence_time,bookmakers
0,New York Knicks,Cleveland Cavaliers,2026-05-20 00:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
1,Oklahoma City Thunder,San Antonio Spurs,2026-05-21 00:40:00+00:00,"[{'bookmaker': 'BetRivers', 'last_updated': '2..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = s26
ast_df = s26
reb_df = s26
min_df = s26


#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
# lines_dfs = lines_dfs[lines_dfs['COMMENCE_TIME'] == current_date]
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
# lines_us = lines_us[lines_us['COMMENCE_TIME'] == current_date]
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]
print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")

lines_dfs_pts.head()

DFS latest pull: 2026-05-19 10:39:24
US latest pull: 2026-05-19 10:39:17


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Underdog,player_points,Jalen Brunson,Over,27.5,-137,2026-05-20,2026-05-19T17:39:05Z,2026-05-19 10:39:24
1,Underdog,player_points,Jalen Brunson,Under,27.5,-137,2026-05-20,2026-05-19T17:39:05Z,2026-05-19 10:39:24
2,Underdog,player_points,Evan Mobley,Over,15.5,-137,2026-05-20,2026-05-19T17:39:05Z,2026-05-19 10:39:24
3,Underdog,player_points,Evan Mobley,Under,15.5,-137,2026-05-20,2026-05-19T17:39:05Z,2026-05-19 10:39:24
4,Underdog,player_points,OG Anunoby,Over,15.5,-137,2026-05-20,2026-05-19T17:39:05Z,2026-05-19 10:39:24


### Load my models

In [44]:
import joblib

#minutes
min_bundle = joblib.load("src\models\saved_models\min_quantile_xgb_v2_2026-05-11.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src\models\saved_models\ppm_quantile_xgb_2026-05-11.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

# #assists per minute
# apm_bundle = joblib.load("src\models\saved_models\apm_quantile_xgb_2026-05-11.joblib")
# apm_quantile_models = apm_bundle["quantile_models"]
# apm_feature_names = apm_bundle["feature_names"]

# #rebounds per minute
# rpm_bundle = joblib.load("src\models\saved_models\rpm_quantile_xgb_2026-05-11.joblib")
# rpm_quantile_models = rpm_bundle["quantile_models"]
# rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [8]:
def_ratings, league_avg_def_rtg, league_avg_pace = load_opp_def_ratings("Playoffs")

Loaded def ratings for 16 teams  |  avg DEF_RTG=112.5  avg PACE=96.2


In [9]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
    def_ratings=def_ratings,
    league_avg_def_rtg=league_avg_def_rtg,
    league_avg_pace=league_avg_pace,
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
    def_ratings=def_ratings,
    league_avg_def_rtg=league_avg_def_rtg,
    league_avg_pace=league_avg_pace,
)
# reb_preds = predict_min_times_rate(
#     reb_names, min_df, reb_df, current_date,
#     rate_pipeline=rpm_pipeline,
#     rate_quantile_models=rpm_quantile_models,
#     min_quantile_models=min_quantile_models,
#     stat_prefix="REB",
#     def_ratings=def_ratings,
#     league_avg_def_rtg=league_avg_def_rtg,
#     league_avg_pace=league_avg_pace,
# )
ast_preds.head(10)

,PLAYER_NAME,PLAYER_TEAM,OPP_TEAM,HOME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,MIN_HISTORY,RATE_Q10,RATE_Q50,RATE_Q90,RATE_HISTORY,OPP_DEF_RATING,OPP_PACE,LEAGUE_AVG_DEF_RATING,LEAGUE_AVG_PACE,STAT_Q10,STAT_Q50,STAT_Q90
0,Mikal Bridges,1610612752,CLE,1,AST,24.70,32.43,38.21,"[20.531666666666663, 19.315, 19.315, 27.483333...",0.0230,0.0993,0.2013,"[0.0974105040993587, 0.0517732332384157, 0.051...",112.6,95.51,112.4625,96.193125,0.57,3.22,7.69
1,Josh Hart,1610612752,CLE,1,AST,25.36,34.13,40.27,"[40.05, 31.048333333333332, 31.048333333333332...",0.0301,0.1016,0.2013,"[0.149812734082397, 0.0966235439368726, 0.0966...",112.6,95.51,112.4625,96.193125,0.76,3.47,8.11
2,James Harden,1610612739,NYK,0,AST,31.22,37.87,41.60,"[44.0, 44.0, 36.63333333333333, 36.63333333333...",0.0885,0.1701,0.3062,"[0.2045454545454545, 0.2045454545454545, 0.081...",104.8,96.45,112.4625,96.193125,2.76,6.44,12.74
3,Donovan Mitchell,1610612739,NYK,0,AST,31.17,37.41,40.56,"[40.961666666666666, 40.961666666666666, 35.19...",0.0396,0.1150,0.2066,"[0.0488261382593481, 0.0488261382593481, 0.028...",104.8,96.45,112.4625,96.193125,1.23,4.30,8.38
4,Evan Mobley,1610612739,NYK,0,AST,27.91,33.83,39.41,"[41.2, 41.2, 27.45, 27.45, 34.766666666666666,...",0.0337,0.0901,0.1892,"[0.0728155339805825, 0.0728155339805825, 0.145...",104.8,96.45,112.4625,96.193125,0.94,3.05,7.46
5,Jalen Brunson,1610612752,CLE,1,AST,29.67,37.83,41.54,"[40.318333333333335, 33.87, 33.87, 34.735, 34....",0.0777,0.1738,0.3040,"[0.0992104501674176, 0.0885739592559787, 0.088...",112.6,95.51,112.4625,96.193125,2.31,6.57,12.63
6,Karl-Anthony Towns,1610612752,CLE,1,AST,21.68,28.34,33.82,"[34.10166666666667, 29.311666666666667, 29.311...",0.0494,0.1381,0.2657,"[0.1172963198279654, 0.3411610848922499, 0.341...",112.6,95.51,112.4625,96.193125,1.07,3.92,8.99
7,De'Aaron Fox,1610612759,OKC,0,AST,27.43,34.99,39.65,"[36.471666666666664, 38.75, 38.75, 33.9, 33.9,...",0.0710,0.1675,0.2890,"[0.1645112644518576, 0.1806451612903225, 0.180...",108.9,95.62,112.4625,96.193125,1.95,5.86,11.46
8,Stephon Castle,1610612759,OKC,0,AST,26.57,35.80,40.06,"[26.266666666666666, 26.266666666666666, 32.96...",0.0986,0.1997,0.3350,"[0.3045685279187817, 0.3045685279187817, 0.151...",108.9,95.62,112.4625,96.193125,2.62,7.15,13.42
9,Shai Gilgeous-Alexander,1610612760,SAS,1,AST,31.76,40.02,43.38,"[38.25, 38.25, 37.93333333333333, 37.933333333...",0.0784,0.1774,0.3047,"[0.2352941176470588, 0.2352941176470588, 0.210...",102.2,99.36,112.4625,96.193125,2.49,7.10,13.22


In [10]:
from live import adjust_predictions

# Build contexts dict once (using the notebook's get_game_context)
game_contexts = {
    name: get_game_context(base_df, name, team_odds, is_playoff=True)
    for name in pts_preds["PLAYER_NAME"]
}

# Adjust the model's Q50 predictions with scenario signals
pts_preds = adjust_predictions(pts_preds, base_df, game_contexts)
# ast_preds = adjust_predictions(ast_preds, base_df, game_contexts)
# reb_preds = adjust_predictions(reb_preds, base_df, game_contexts)
pts_preds.head()

Jalen Brunson [PTS] pace=low_pace def=strong_def spread=favorite stars=3 MIN: 37.8→35.4 (Δ-2.88) RATE: 0.7351→0.7450 (+2.1%)
Evan Mobley [PTS] pace=low_pace def=weak_def spread=underdog stars=3 MIN: 33.8→32.3 (Δ-1.76) RATE: 0.4890→0.5271 (+12.0%)
OG Anunoby [PTS] pace=low_pace def=strong_def spread=favorite stars=3 MIN: 36.4→36.9 (Δ+0.58) RATE: 0.4621→0.4726 (+3.5%)
Josh Hart [PTS] pace=low_pace def=strong_def spread=favorite stars=3 MIN: 34.1→35.4 (Δ+1.50) RATE: 0.3671→0.3391 (-11.7%)
Mitchell Robinson [PTS] pace=low_pace def=mid_def spread=favorite stars=3 MIN: 15.6→18.2 (Δ+3.00) RATE: 0.3559→0.3281 (-12.0%)
Jose Alvarado [PTS] pace=low_pace def=mid_def spread=favorite stars=3 MIN: 10.6→13.2 (Δ+3.00) RATE: 0.4269→0.3936 (-12.0%)
Victor Wembanyama [PTS] pace=low_pace def=strong_def spread=underdog stars=3 MIN: 34.2→31.7 (Δ-3.00) RATE: 0.6798→0.7328 (+12.0%)
De'Aaron Fox [PTS] pace=low_pace def=strong_def spread=underdog stars=3 MIN: 35.0→33.2 (Δ-2.05) RATE: 0.5999→0.5616 (-9.8%)
Jalen

,PLAYER_NAME,PLAYER_TEAM,OPP_TEAM,HOME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,MIN_HISTORY,RATE_Q10,RATE_Q50,RATE_Q90,RATE_HISTORY,OPP_DEF_RATING,OPP_PACE,LEAGUE_AVG_DEF_RATING,LEAGUE_AVG_PACE,STAT_Q10,STAT_Q50,STAT_Q90
0,Jalen Brunson,1610612752,CLE,1,PTS,26.79,35.38,38.66,"[40.318333333333335, 33.87, 33.87, 34.735, 34....",0.4638,0.7450,1.0336,"[0.653539, 0.568511, 0.568511, 1.137884, 1.137...",112.6,95.51,112.4625,96.193125,12.43,26.36,39.96
1,Evan Mobley,1610612739,NYK,0,PTS,26.15,32.33,37.65,"[41.2, 41.2, 27.45, 27.45, 34.766666666666666,...",0.2979,0.5271,0.7578,"[0.680291, 0.680291, 0.510528, 0.510528, 0.434...",104.8,96.45,112.4625,96.193125,7.79,17.04,28.53
2,OG Anunoby,1610612752,CLE,1,PTS,27.86,36.86,41.22,"[37.93333333333333, 37.9, 37.9, 36.56666666666...",0.2543,0.4726,0.7847,"[0.485325, 0.377807, 0.377807, 0.811136, 0.811...",112.6,95.51,112.4625,96.193125,7.08,17.42,32.35
3,Josh Hart,1610612752,CLE,1,PTS,26.86,35.40,41.77,"[40.05, 31.048333333333332, 31.048333333333332...",0.1559,0.3391,0.5657,"[0.046127, 0.2975, 0.2975, 0.274967, 0.274967,...",112.6,95.51,112.4625,96.193125,4.19,12.00,23.63
4,Mitchell Robinson,1610612752,CLE,1,PTS,12.13,18.16,24.37,"[18.15, 11.383333333333333, 11.383333333333333...",0.0947,0.3281,0.5721,"[0.660386, 0.161991, 0.161991, 0.366236, 0.366...",112.6,95.51,112.4625,96.193125,1.15,5.96,13.94


### Get Line Probabilities

In [12]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, run_pts_simulation),
    # line_probs_for_market(reb_preds, lines_dfs_reb, run_pts_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, run_pts_simulation),
], ignore_index=True)
all_line_probs.head()

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
0,Mikal Bridges,AST,2.5,24.70,32.43,38.21,0.57,3.22,7.69,0.580,0.420
1,Josh Hart,AST,4.5,25.36,34.13,40.27,0.76,3.47,8.11,0.333,0.667
2,James Harden,AST,6.5,31.22,37.87,41.60,2.76,6.44,12.74,0.396,0.604
3,Donovan Mitchell,AST,4.0,31.17,37.41,40.56,1.23,4.30,8.38,0.302,0.698
4,Evan Mobley,AST,3.5,27.91,33.83,39.41,0.94,3.05,7.46,0.704,0.296


In [13]:
import pandas as pd
from src.utils.generalized_best_bets_v2 import enrich_dfs_picks

dfs_df = pd.read_csv("data/raw/player_lines/NBA_DFS_20260510_213929.csv")
us_df  = pd.read_csv("data/raw/player_lines/NBA_US_20260510_213820.csv")

enriched_path, aligned_path, df = enrich_dfs_picks(
    dfs_df=dfs_df,
    us_df=us_df,
    base_df=base_df,                  # your existing game log df
    all_line_probs=all_line_probs,            # your existing all_line_probs df
    team_odds_source="data/props/circa+betonline_team_lines/circa+betonline_20260510_213347.json",
    dfs_platforms=None,  # or None for all four
    current_date="2026-05-10",
)


  DFS rows: 757 across ['Betr DFS', 'DraftKings Pick6', 'PrizePicks', 'Underdog']
  Sharp books available: ['Pinnacle', 'FanDuel', 'DraftKings', 'BetMGM', 'BetOnline.ag', 'Bovada']
  Fetching league team ratings (NBA API)...
  Enriched: 703 | sharp_verified: 103 | dfs_only: 0 | conflict: 101 | no_model: 499
  → data/props/enriched/dfs_enriched_20260510.json
  → data/props/enriched/dfs_sharp_aligned_20260510.json


In [10]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
underdog_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
underdog_all_lines.head()

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
7,Joel Embiid,AST,4.5,26.26,31.36,40.54,15.40,17.41,20.49,0.757,0.243,AST,Underdog,New York Knicks,-1.5,214.5,112.3,7.0,97.71,25.0,-110.0,-105.0,0.524,0.512,4.9,6.0,2.85,0.4,1.5,-0.140,0.556,0.444,6.15,-13.31,0.8,0.6,0.53,0.40,34.24,4.76,0.34,0.04,4.00,4.0
28,Quentin Grimes,REB,2.5,15.05,21.94,27.71,7.52,10.15,10.91,0.732,0.268,REB,Underdog,New York Knicks,-1.5,214.5,112.3,7.0,97.71,25.0,-110.0,-105.0,0.524,0.512,3.1,3.0,1.37,0.6,0.5,-0.438,0.669,0.331,27.72,-35.38,0.4,0.7,0.67,0.69,22.52,4.15,0.14,0.05,3.29,7.0
30,Rudy Gobert,REB,11.5,27.60,31.78,38.88,18.02,20.47,23.75,0.589,0.411,REB,Underdog,San Antonio Spurs,4.5,216.5,110.4,3.0,100.72,12.0,102.0,-115.0,0.495,0.535,11.2,11.0,3.01,-0.3,-0.5,0.100,0.460,0.540,-7.08,0.96,0.6,0.5,0.53,0.46,33.07,4.64,0.11,0.04,9.57,7.0
38,De'Aaron Fox,REB,3.5,27.44,32.93,37.80,12.94,14.55,13.92,0.569,0.431,REB,Underdog,Minnesota Timberwolves,-4.5,216.5,112.5,8.0,101.50,10.0,-115.0,100.0,0.535,0.500,3.5,3.5,2.07,0.0,0.0,0.000,0.500,0.500,-6.52,0.00,0.4,0.5,0.53,0.57,33.18,3.68,0.25,0.03,3.88,8.0
62,Joel Embiid,PTS,26.5,26.26,31.36,40.54,13.36,23.81,40.82,0.464,0.536,PTS,Underdog,New York Knicks,-1.5,214.5,112.3,7.0,97.71,25.0,-103.0,-110.0,0.507,0.524,26.9,27.5,7.46,0.4,1.0,-0.054,0.522,0.478,2.88,-8.75,0.4,0.5,0.53,0.55,34.24,4.76,0.34,0.04,22.75,4.0


In [11]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
prizePicks_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
prizePicks_all_lines.head(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
0,Shai Gilgeous-Alexander,AST,4.5,31.48,37.38,40.08,19.72,23.02,23.74,0.951,0.049,AST,PrizePicks,Los Angeles Lakers,-8.5,215.2,115.5,20.0,99.22,22.0,100.0,108.0,0.500,0.481,7.4,7.5,2.07,2.9,3.0,-1.401,0.919,0.081,83.80,-83.15,1.0,0.9,0.87,0.80,33.04,5.57,0.32,0.04,7.71,7.0
1,Austin Reaves,AST,3.5,32.59,37.79,41.92,17.38,19.78,19.16,0.711,0.289,AST,PrizePicks,Oklahoma City Thunder,8.5,215.2,106.5,1.0,100.37,16.0,-115.0,110.0,0.535,0.476,5.1,5.0,2.42,1.6,1.5,-0.661,0.746,0.254,39.47,-46.66,0.6,0.8,0.87,0.71,34.86,5.12,0.26,0.04,4.00,7.0
2,Luguentz Dort,AST,1.5,21.03,27.97,33.23,10.66,13.52,14.66,0.687,0.313,AST,PrizePicks,Los Angeles Lakers,-8.5,215.2,115.5,20.0,99.22,22.0,-115.0,-115.0,0.535,0.535,1.5,1.5,1.35,0.0,0.0,0.000,0.500,0.500,-6.52,-6.52,0.6,0.5,0.40,0.38,23.03,2.75,0.12,0.03,1.86,7.0
3,Jalen Brunson,AST,6.5,30.83,34.60,40.53,17.65,19.39,21.15,0.629,0.371,AST,PrizePicks,Philadelphia 76ers,1.5,214.5,114.4,17.0,100.39,15.0,-137.0,-137.0,0.578,0.578,6.5,7.0,3.50,0.5,1.0,-0.143,0.557,0.443,-3.64,-23.36,0.4,0.6,0.67,0.56,34.74,3.70,0.32,0.05,5.25,8.0
4,Tyrese Maxey,AST,6.0,26.14,32.91,43.25,14.86,17.89,21.84,0.502,0.498,AST,PrizePicks,New York Knicks,-1.5,214.5,112.3,7.0,97.71,25.0,-137.0,-137.0,0.578,0.578,5.4,5.5,2.46,-0.6,-0.5,0.244,0.404,0.596,-30.11,3.10,0.2,0.3,0.40,0.46,37.86,6.29,0.27,0.05,4.86,7.0


In [13]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
betr_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
93,James Harden,PTS,18.5,31.47,37.88,42.27,9.94,20.40,36.28,0.537,0.463,PTS,Betr DFS,Toronto Raptors,-8.5,210.5,112.1,5.0,99.22,21.0,-122.0,-103.0,0.550,0.507,21.3,20.5,4.08,2.8,2.0,-0.686,0.754,0.246,37.20,-51.52,0.4,0.7,0.53,0.71,35.19,4.93,0.27,0.06,22.40,10.0
111,Ajay Mitchell,PTS,15.5,23.68,33.18,38.39,4.29,14.08,26.24,0.366,0.634,PTS,Betr DFS,Los Angeles Lakers,-15.8,213.5,115.5,20.0,99.22,22.0,-115.0,-106.0,0.535,0.515,11.5,9.5,4.74,-4.0,-6.0,0.844,0.199,0.801,-62.80,55.67,0.2,0.1,0.13,0.22,25.78,6.70,0.19,0.06,9.25,4.0
39,Evan Mobley,REB,8.5,25.64,33.17,39.57,3.52,7.64,13.62,0.503,0.497,REB,Betr DFS,Toronto Raptors,-8.5,210.5,112.1,5.0,99.22,21.0,-113.0,-103.0,0.531,0.507,9.1,7.5,4.23,0.6,-1.0,-0.142,0.556,0.444,4.80,-12.49,0.6,0.4,0.47,0.56,31.69,5.46,0.22,0.05,9.14,14.0
28,Julius Randle,REB,6.5,26.56,35.36,40.93,1.65,5.43,11.24,0.430,0.570,REB,Betr DFS,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,110.0,-130.0,0.476,0.565,6.7,7.0,2.31,0.2,0.5,-0.087,0.535,0.465,12.35,-17.73,0.6,0.6,0.53,0.54,33.35,2.97,0.27,0.04,6.29,7.0
119,Cason Wallace,PTS,6.5,9.53,15.75,22.80,0.73,5.51,14.92,0.305,0.695,PTS,Betr DFS,Los Angeles Lakers,-15.8,213.5,115.5,20.0,99.22,22.0,-128.0,105.0,0.561,0.488,6.8,6.0,4.34,0.3,-0.5,-0.069,0.528,0.472,-5.95,-3.24,0.2,0.4,0.40,0.52,21.87,3.16,0.14,0.05,7.43,7.0


In [14]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
draftKings_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
25,Quentin Grimes,REB,2.5,12.27,19.21,27.07,0.49,2.12,5.94,0.565,0.435,REB,DraftKings Pick6,New York Knicks,7.5,212.5,112.3,7.0,97.71,25.0,130.0,-115.0,0.435,0.535,3.2,3.0,1.23,0.7,0.5,-0.569,0.715,0.285,64.45,-46.72,0.6,0.7,0.73,0.69,22.74,4.31,0.17,0.07,3.67,6.0
26,Rudy Gobert,REB,11.5,21.35,30.40,36.47,2.47,6.47,13.26,0.126,0.874,REB,DraftKings Pick6,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,110.0,-116.0,0.476,0.537,10.9,11.0,3.28,-0.6,-0.5,0.183,0.427,0.573,-10.33,6.70,0.6,0.5,0.60,0.46,32.82,4.90,0.11,0.04,9.50,6.0
72,OG Anunoby,PTS,14.5,27.95,35.76,41.20,5.32,15.77,29.23,0.638,0.362,PTS,DraftKings Pick6,Philadelphia 76ers,-7.5,212.5,114.4,17.0,100.40,15.0,100.0,-130.0,0.500,0.565,19.7,20.0,8.90,5.2,5.5,-0.584,0.720,0.280,44.00,-50.46,0.8,0.7,0.67,0.64,33.30,7.34,0.18,0.05,18.29,7.0
3,Julius Randle,AST,4.5,26.56,35.36,40.93,0.66,3.65,6.91,0.297,0.703,AST,DraftKings Pick6,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,129.0,-135.0,0.437,0.574,4.2,4.0,1.55,-0.3,-0.5,0.194,0.423,0.577,-3.13,0.44,0.6,0.4,0.33,0.47,33.35,2.97,0.27,0.04,5.14,7.0
77,Anthony Edwards,PTS,20.5,16.76,23.86,32.55,2.64,9.12,23.30,0.103,0.897,PTS,DraftKings Pick6,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,115.0,-122.0,0.465,0.550,21.8,20.5,11.56,0.3,-1.0,-0.026,0.510,0.490,9.65,-10.84,0.6,0.5,0.53,0.72,30.48,7.93,0.31,0.04,28.00,7.0


In [15]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
51,Isaiah Hartenstein,REB,8.5,9.32,16.88,23.18,1.59,5.15,10.54,0.259,0.741,REB,PrizePicks,Los Angeles Lakers,-15.8,213.5,115.5,20.0,99.22,22.0,-137.0,-137.0,0.578,0.578,8.8,8.0,3.97,0.8,0.0,-0.202,0.580,0.420,0.34,-27.34,0.4,0.4,0.47,0.56,21.14,4.87,0.14,0.03,9.57,7.0
41,Ausar Thompson,REB,7.0,24.75,32.87,39.53,3.14,7.07,12.91,0.660,0.340,REB,Betr DFS,Orlando Magic,-8.5,202.0,113.6,13.0,100.56,14.0,-137.0,-137.0,0.578,0.578,7.5,7.5,3.37,0.5,0.5,-0.148,0.559,0.441,-3.30,-23.71,0.8,0.5,0.40,0.23,30.03,5.87,0.14,0.04,7.92,13.0
44,Donovan Mitchell,REB,4.0,30.13,36.45,41.75,2.12,4.86,9.86,0.803,0.197,REB,PrizePicks,Toronto Raptors,-8.5,210.5,112.1,5.0,99.22,21.0,-137.0,-137.0,0.578,0.578,5.3,5.5,1.42,1.3,1.5,-0.915,0.820,0.180,41.85,-68.86,0.8,0.8,0.60,0.46,34.73,3.00,0.30,0.06,4.42,12.0
114,Marcus Smart,PTS,10.5,30.59,39.25,45.10,5.65,17.72,33.50,0.766,0.234,PTS,Underdog,Oklahoma City Thunder,15.8,213.5,106.5,1.0,100.37,16.0,105.0,-115.0,0.488,0.535,11.4,10.0,7.28,0.9,-0.5,-0.124,0.549,0.451,12.54,-15.68,0.6,0.5,0.47,0.39,31.33,6.19,0.18,0.06,14.00,2.0
29,Naz Reid,REB,6.5,17.35,24.38,30.98,1.52,4.71,10.71,0.379,0.621,REB,Betr DFS,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,130.0,-140.0,0.435,0.583,6.3,7.0,2.36,-0.2,0.5,0.085,0.466,0.534,7.18,-8.46,0.8,0.6,0.53,0.39,24.88,4.94,0.21,0.03,5.57,7.0


### Get top EVs for 2 legs

In [16]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 87  |  Pairs: 885  |  Slate: 10  |  STRONG: 1  |  MARGINAL: 9  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks.json


In [17]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 26  |  Pairs: 105  |  Slate: 6  |  STRONG: 0  |  MARGINAL: 6  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog.json


In [18]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 33  |  Pairs: 90  |  Slate: 6  |  STRONG: 0  |  MARGINAL: 6  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings.json


In [19]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 77  |  Pairs: 655  |  Slate: 10  |  STRONG: 0  |  MARGINAL: 10  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr.json


### Top EVs for 3 Legs

In [20]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 87  |  Triples: 18373  |  Slate: 10  |  STRONG: 0  |  MARGINAL: 9  |  SKIP: 1  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks_3leg.json


In [21]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 26  |  Triples: 623  |  Slate: 5  |  STRONG: 0  |  MARGINAL: 5  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog_3leg.json


In [22]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 77  |  Triples: 12278  |  Slate: 9  |  STRONG: 0  |  MARGINAL: 9  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr_3leg.json


In [23]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

Legs: 33  |  Triples: 502  |  Slate: 4  |  STRONG: 0  |  MARGINAL: 3  |  SKIP: 1  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings_3leg.json
